# Step 10c-1: truy xuất 4 danh sách cho 4 bộ mới + quét lưới trọng số

Dùng nguyên `run_retrieval.py` (Step 2) và `grid_sweep.py` (Step 3).
Các split: SciFact test + train, NFCorpus test + train, DBPedia-entity test, FEVER test + 10.000 query train đầu (cùng cách cắt với expansion ở Step 10b).

- **Accelerator:** GPU T4 ×2 (GPU chỉ dùng cho dense search; BM25 chạy trên CPU). **Internet:** On (clone QuDAR, tải `facebook/contriever`).
- **Input:** `qudar-beir-data-2`, `qudar-encode-new`, `qudar-shards-fever`, `qudar-expansions-2`, `qudar-patches`.
- **Thời gian ước tính** (theo log Step 2): cài đặt khoảng 5 phút; SciFact + NFCorpus 5–10 phút; DBPedia-entity 20–30 phút; FEVER khoảng 3 giờ (index 10 phút, BM25 33.332 lượt tìm × khoảng 0,23 giây ≈ 2 giờ, dense 15 phút, đánh giá 35 phút); quét lưới 20–40 phút. **Tổng khoảng 4–5 giờ**, tốn chừng đó quota GPU.
- **Cách chạy:** Save Version → **Save & Run All (Commit)** để chạy nền, không cần giữ trình duyệt (Kaggle cho tối đa 12 giờ một phiên).
- **Sau khi chạy:** (1) tạo dataset `qudar-runs-2` từ thư mục `runs/` (dùng cho bước chấm LLM sau); (2) tải `for_repo_step10c1.zip` về, giải nén vào gốc repo (ra `results/step10/runs/` và `results/step10/grid/`); (3) gửi lại bảng ở ô cuối.

In [1]:
import glob, os, shutil
def first(pattern, what):
    hits = sorted(glob.glob(pattern, recursive=True))
    assert hits, f"không tìm thấy {what} ({pattern}): kiểm tra dataset đã gắn chưa"
    return hits[0]
DATA2  = os.path.dirname(os.path.dirname(first("/kaggle/input/**/qudar-beir-data-2/**/dbpedia-entity/corpus.jsonl", "qudar-beir-data-2")))
PATCH  = os.path.dirname(first("/kaggle/input/**/run_retrieval.py", "qudar-patches"))
SHARDS = {ds: os.path.dirname(first(f"/kaggle/input/**/qudar-encode-new/**/{ds}/shard_000.npy", f"shard {ds} trong qudar-encode-new"))
          for ds in ["scifact", "nfcorpus", "dbpedia-entity"]}
SHARDS["fever"] = os.path.dirname(first("/kaggle/input/**/qudar-shards-fever/**/shard_000.npy", "shard trong qudar-shards-fever"))
EXP = "/kaggle/working/exp"
os.makedirs(EXP, exist_ok=True)
exp_files = sorted(glob.glob("/kaggle/input/**/qudar-expansions-2/**/expanded_queries_*_expansion.json", recursive=True))
for f in exp_files:
    shutil.copy(f, EXP)
print("DATA2 =", DATA2, "\nPATCH =", PATCH)
for ds, d in SHARDS.items():
    print(f"SHARDS[{ds}] = {d} | {len(glob.glob(d + '/shard_*[0-9].npy'))} shard")
print(sorted(os.listdir(EXP)))
assert len(os.listdir(EXP)) == 7, "cần đủ 7 file expansion (scifact/nfcorpus test+train, fever test+train, dbpedia-entity test)"
!git clone -q https://github.com/kaist-dmlab/QuDAR.git /kaggle/working/QuDAR
!pip install -q -e /kaggle/working/QuDAR bm25s
!python {PATCH}/apply_patch.py /kaggle/working/QuDAR
!grep -c backend_selection {PATCH}/bm25_fast.py
RUN = f"python {PATCH}/run_retrieval.py --repo /kaggle/working/QuDAR --data-root {DATA2} --expansions {EXP} --out /kaggle/working/runs"

DATA2 = /kaggle/input/datasets/karrychuong/qudar-beir-data-2/data 
PATCH = /kaggle/input/datasets/karrychuong/qudar-patches/qudar-patches
SHARDS[scifact] = /kaggle/input/datasets/karrychuong/qudar-encode-new/shards/scifact | 1 shard
SHARDS[nfcorpus] = /kaggle/input/datasets/karrychuong/qudar-encode-new/shards/nfcorpus | 1 shard
SHARDS[dbpedia-entity] = /kaggle/input/datasets/karrychuong/qudar-encode-new/shards/dbpedia-entity | 10 shard
SHARDS[fever] = /kaggle/input/datasets/karrychuong/qudar-shards-fever/kaggle/working/shards/fever | 12 shard
['expanded_queries_dbpedia-entity_test_expansion.json', 'expanded_queries_fever_test_expansion.json', 'expanded_queries_fever_train_expansion.json', 'expanded_queries_nfcorpus_test_expansion.json', 'expanded_queries_nfcorpus_train_expansion.json', 'expanded_queries_scifact_test_expansion.json', 'expanded_queries_scifact_train_expansion.json']
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Ge

In [2]:
# SciFact và NFCorpus: chạy để kiểm tra nhanh cả pipeline
S = SHARDS["scifact"]
!{RUN} --dataset scifact --splits test train --dense-shards {S}
S = SHARDS["nfcorpus"]
!{RUN} --dataset nfcorpus --splits test train --dense-shards {S}

[15:22:55] scifact/test: 300 queries
[15:22:55] scifact/train: 809 queries
[15:22:55] BM25: reading corpus
[15:22:56] BM25: indexing 5183 documents
[15:23:14] BM25 search test: OS + ES
[15:23:14] BM25 search train: OS + ES
[15:23:41] Shard check: 5183 shard ids vs 5183 corpus docs, 0/2592 sampled ids missing
tokenizer_config.json: 100%|███████████████████| 321/321 [00:00<00:00, 1.66MB/s]
vocab.txt: 100%|█████████████████████████████| 232k/232k [00:00<00:00, 14.4MB/s]
tokenizer.json: 100%|████████████████████████| 466k/466k [00:00<00:00, 37.9MB/s]
special_tokens_map.json: 100%|██████████████████| 112/112 [00:00<00:00, 634kB/s]

pytorch_model.bin: downloading bytes: ███████████▍          |  228MB, 16.8MB/s  
pytorch_model.bin: reconstructing file:  15%|▉     | 67.1MB /  438MB            
pytorch_model.bin: downloading bytes: ███████████████████▎  |  383MB, 33.6MB/s  
pytorch_model.bin: downloading bytes: ██████████████████████|  413MB, 37.4MB/s  
pytorch_model.bin: reconstructing file: 1

In [3]:
# DBPedia-entity (4,6 triệu tài liệu, 400 query test)
S = SHARDS["dbpedia-entity"]
!{RUN} --dataset dbpedia-entity --splits test --dense-shards {S}

[15:29:54] dbpedia-entity/test: 400 queries
[15:29:54] BM25: reading corpus
[15:30:25] BM25: indexing 4635922 documents
[15:36:10] BM25 search test: OS + ES
[15:39:14] Shard check: 4635922 shard ids vs 4635922 corpus docs, 0/2001 sampled ids missing
Loading weights: 100%|█████████████████████| 199/199 [00:00<00:00, 10520.92it/s]
[15:39:16] Dense test: encode queries
Dense encoding: 100%|█████████████████████████████| 2/2 [00:05<00:00,  2.56s/it]
[15:39:30]   dense search: shard_000.npy done (500000 docs)
[15:39:38]   dense search: shard_001.npy done (1000000 docs)
[15:39:43]   dense search: shard_002.npy done (1500000 docs)
[15:39:50]   dense search: shard_003.npy done (2000000 docs)
[15:39:54]   dense search: shard_004.npy done (2500000 docs)
[15:39:59]   dense search: shard_005.npy done (3000000 docs)
[15:40:03]   dense search: shard_006.npy done (3500000 docs)
[15:40:09]   dense search: shard_007.npy done (4000000 docs)
[15:40:12]   dense search: shard_008.npy done (4500000 docs)
[1

In [4]:
# FEVER (5,4 triệu tài liệu): test 6.666 query + 10.000 query train đầu
S = SHARDS["fever"]
!{RUN} --dataset fever --splits test train:10000 --dense-shards {S}

[15:41:01] fever/test: 6666 queries
[15:41:02] fever/train: 10000 queries
[15:41:02] BM25: reading corpus
[15:41:47] BM25: indexing 5416568 documents
[15:52:25] BM25 search test: OS + ES
[16:43:44] BM25 search train: OS + ES
[18:00:31] Shard check: 5416568 shard ids vs 5416568 corpus docs, 0/2001 sampled ids missing
Loading weights: 100%|█████████████████████| 199/199 [00:00<00:00, 10075.04it/s]
[18:00:33] Dense test: encode queries
Dense encoding: 100%|███████████████████████████| 27/27 [01:37<00:00,  3.59s/it]
[18:02:24]   dense search: shard_000.npy done (500000 docs)
[18:02:28]   dense search: shard_001.npy done (1000000 docs)
[18:02:31]   dense search: shard_002.npy done (1500000 docs)
[18:02:35]   dense search: shard_003.npy done (2000000 docs)
[18:02:39]   dense search: shard_004.npy done (2500000 docs)
[18:02:43]   dense search: shard_005.npy done (3000000 docs)
[18:02:47]   dense search: shard_006.npy done (3500000 docs)
[18:02:51]   dense search: shard_007.npy done (4000000 d

In [5]:
# Quét lưới trọng số 0,1 (cận trên tĩnh và theo từng query)
NAMES = "scifact_test scifact_train nfcorpus_test nfcorpus_train dbpedia-entity_test fever_test fever_train"
!python {PATCH}/grid_sweep.py --runs /kaggle/working/runs --data-root {DATA2} --out /kaggle/working/grid --names {NAMES}


=== scifact_test (300 queries) ===
                       axis             method  Recall@10  nDCG@10  Precision@1 best (a, b1, b2)  max |diff| vs eval_results
     retriever_type (OS-OD)         Static Avg     0.8215   0.6818       0.5421              NaN                         0.0
     retriever_type (OS-OD)          Static UB     0.8602   0.7283       0.6000  (0.4, 0.0, 0.0)                         0.0
     retriever_type (OS-OD)      Query-wise UB     0.8952   0.7934       0.6933              NaN                         NaN
query_format_sparse (OS-ES)         Static Avg     0.8279   0.6769       0.5261              NaN                         0.0
query_format_sparse (OS-ES)          Static UB     0.8532   0.7078       0.5700  (0.0, 0.4, 0.0)                         0.0
query_format_sparse (OS-ES)      Query-wise UB     0.8826   0.7617       0.6400              NaN                         NaN
 query_format_dense (OD-ED)         Static Avg     0.7063   0.5447       0.3967          

In [6]:
# Tóm tắt + đóng gói file nhỏ cho repo (không gồm retrieval_result.json)
import pandas as pd
pd.set_option("display.width", 250)
rows = []
for name in NAMES.split():
    ev = pd.read_csv(f"/kaggle/working/runs/{name}/eval_results.csv").set_index("method")
    gr = pd.read_csv(f"/kaggle/working/grid/{name}/summary.csv")
    dual = gr[gr["axis"] == "dual (4 lists)"].set_index("method")
    chk = gr[gr["axis"] == "check"]["max |diff| vs eval_results"].max()
    r = {"run": name}
    for m, lab in [("os", "OS"), ("od", "OD"), ("es", "ES"), ("ed", "ED"), ("qudar_simple_equal", "Equal"), ("qudar_confidence", "Conf")]:
        r[lab] = ev.loc[m, "Recall@10"]
    r["ES/OS-1"] = round(ev.loc["es", "Recall@10"] / ev.loc["os", "Recall@10"] - 1, 3)
    r["| Equal (nDCG)"] = ev.loc["qudar_simple_equal", "nDCG@10"]
    r["Conf (nDCG)"] = ev.loc["qudar_confidence", "nDCG@10"]
    r["Static UB (nDCG)"] = dual.loc["Static UB", "nDCG@10"]
    r["Query UB (nDCG)"] = dual.loc["Query-wise UB", "nDCG@10"]
    r["check max|diff|"] = chk
    rows.append(r)
print("Bên trái: Recall@10 của 4 danh sách và 2 cách trộn; bên phải (sau dấu |): nDCG@10, gồm cận trên 4 danh sách")
print(pd.DataFrame(rows).to_string(index=False))

import shutil
from pathlib import Path
dst = Path("/kaggle/working/for_repo/results/step10")
for name in NAMES.split():
    for f in Path(f"/kaggle/working/runs/{name}").iterdir():
        if f.name != "retrieval_result.json":
            (dst / "runs" / name).mkdir(parents=True, exist_ok=True); shutil.copy(f, dst / "runs" / name / f.name)
    shutil.copytree(f"/kaggle/working/grid/{name}", dst / "grid" / name, dirs_exist_ok=True)
shutil.make_archive("/kaggle/working/for_repo_step10c1", "zip", "/kaggle/working/for_repo")
!du -sh /kaggle/working/runs /kaggle/working/for_repo_step10c1.zip

Bên trái: Recall@10 của 4 danh sách và 2 cách trộn; bên phải (sau dấu |): nDCG@10, gồm cận trên 4 danh sách
                run     OS     OD     ES     ED  Equal   Conf  ES/OS-1  | Equal (nDCG)  Conf (nDCG)  Static UB (nDCG)  Query UB (nDCG)  check max|diff|
       scifact_test 0.8187 0.7176 0.7826 0.6256 0.8649 0.8643   -0.044          0.7231       0.7240            0.7418           0.8454           0.0000
      scifact_train 0.8137 0.7234 0.7435 0.5956 0.8500 0.8513   -0.086          0.7168       0.7198            0.7417           0.8364           0.0000
      nfcorpus_test 0.1471 0.1244 0.1394 0.0881 0.1726 0.1725   -0.052          0.3528       0.3527            0.3655           0.4499           0.0031
     nfcorpus_train 0.1418 0.1157 0.1308 0.0739 0.1621 0.1629   -0.078          0.3435       0.3439            0.3539           0.4529           0.0011
dbpedia-entity_test 0.1924 0.1789 0.1432 0.1793 0.2595 0.2591   -0.256          0.3978       0.3984            0.4060           0.53